# Encoding Numerical Data to Categorical Data

## Discretization or Binning 

### Definition:
- Discretization (also called binning) is the process of converting continuous numerical variables into discrete categories or intervals (bins). This can simplify models, reduce noise, and make certain algorithms work better.

### Purpose:
- Handle continuous data in algorithms that require categorical input (e.g., Naive Bayes).
- Reduce the effect of minor observation errors or noise.
- Improve interpretability of data.
- Create meaningful categories from continuous values.

### Advantages of Discretization
- Reduces the impact of minor measurement errors.
- Simplifies models and improves interpretability.
- Can help in handling non-linear relationships in certain algorithms.
### Disadvantages
- Loss of information due to reduced precision.
- Can introduce arbitrary boundaries.
- Poorly chosen bins can lead to misleading results.

## Binning or Discretization 
Are 3 type 
- Unsupervised Binning
- Supervised Binning
- Custom Binning 

### Unsupervised Binning
Are also 3 types 

#### 1. Equal-width Binning (Uniform Binning)
- Divide the range of the variable into k equal-sized intervals.
- Only considers min and max values.
- Pros: Simple to implement.
- Cons: Bins may have very unequal numbers of points if data is skewed.

Example: Ages 0–100 → bins of width 20 → [0–20, 21–40, …].

#### 2. Equal-frequency Binning (Quantile Binning)
- Divide the data into k bins containing roughly the same number of points.
- Pros: Avoids empty or sparse bins.
- Cons: Bin width varies; can merge dissimilar values.

Example: Top 25% scores → Q1, next 25% → Q2, etc.

#### 3. Clustering-based Binning
- Uses clustering algorithms (like K-Means) to group similar values together.
- Pros: Bins are data-driven and capture natural groupings.
- Cons: More computationally expensive; may vary depending on initialization.

Example: K-Means clustering on feature values, each cluster → a bin.

In [45]:
import numpy as np
import pandas as pd

In [46]:
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score

from sklearn.compose import ColumnTransformer

In [47]:
df = pd.read_csv('/home/ipartzix/AI-ML-ENGINEERING-JOURNEY/10_Projects/02_Titanic_EDA_Project/data/raw/train.csv')[['Age','Fare','SibSp','Parch','Survived']]

In [48]:
df.dropna(inplace=True)

In [49]:
df.head()

,Age,Fare,SibSp,Parch,Survived
0,22.0,7.2500,1,0,0
1,38.0,71.2833,1,0,1
2,26.0,7.9250,0,0,1
3,35.0,53.1000,1,0,1
4,35.0,8.0500,0,0,0


In [50]:
df['family'] = df['SibSp'] + df['Parch']

In [51]:
df.head()

,Age,Fare,SibSp,Parch,Survived,family
0,22.0,7.2500,1,0,0,1
1,38.0,71.2833,1,0,1,1
2,26.0,7.9250,0,0,1,0
3,35.0,53.1000,1,0,1,1
4,35.0,8.0500,0,0,0,0


In [52]:
df.drop(columns=['SibSp','Parch'],inplace=True)

In [53]:
df.head()

,Age,Fare,Survived,family
0,22.0,7.2500,0,1
1,38.0,71.2833,1,1
2,26.0,7.9250,1,0
3,35.0,53.1000,1,1
4,35.0,8.0500,0,0


In [54]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [55]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [56]:
X_train.head()

,Age,Fare,family
328,31.0,20.5250,2
73,26.0,14.4542,1
253,30.0,16.1000,1
719,33.0,7.7750,0
666,25.0,13.0000,0


In [57]:
# Without binarization

clf = DecisionTreeClassifier()

clf.fit(X_train,y_train)

y_pred = clf.predict(X_test)

accuracy_score(y_test,y_pred)

0.6293706293706294

In [58]:
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy'))

np.float64(0.6457159624413146)

In [59]:

# Applying Binarization

from sklearn.preprocessing import Binarizer

In [60]:
trf = ColumnTransformer([
    ('bin',Binarizer(copy=False),['family'])
],remainder='passthrough')

In [61]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

In [62]:
pd.DataFrame(X_train_trf,columns=['family','Age','Fare']) # type: ignore

,family,Age,Fare
0,1.0,31.0,20.5250
1,1.0,26.0,14.4542
2,1.0,30.0,16.1000
3,0.0,33.0,7.7750
4,0.0,25.0,13.0000
...,...,...,...
566,1.0,46.0,61.1750
567,0.0,25.0,13.0000
568,0.0,41.0,134.5000
569,1.0,33.0,20.5250


In [63]:

clf = DecisionTreeClassifier()
clf.fit(X_train_trf,y_train)
y_pred2 = clf.predict(X_test_trf)

accuracy_score(y_test,y_pred2)

0.6083916083916084

In [64]:
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeClassifier(),X_trf,y,cv=10,scoring='accuracy'))

np.float64(0.6289906103286385)